<a href="https://colab.research.google.com/github/prasathr0811/Image-caption-generator/blob/main/Image_Captioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install necessary libraries


In [ ]:
!pip install -q transformers accelerate torchvision matplotlib scikit-learn

# Import Necessary Libraries


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch
from google.colab import files
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Load model and processor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

# Upload image

In [ ]:
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert('RGB')

# Display image

In [ ]:
plt.imshow(image)
plt.axis('off')
plt.title("Uploaded Image")
plt.show()

# Generate 5 diverse captions

In [ ]:
inputs = processor(images=image, return_tensors="pt").to(device)
num_captions = 5
captions = []

for i in range(num_captions):
    output = model.generate(
        **inputs,
        max_length=40,
        num_return_sequences=1,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=1.0,
        repetition_penalty=1.2
    )
    caption = processor.decode(output[0], skip_special_tokens=True)
    captions.append(caption)
    print(f"🔹 Caption {i+1}: {caption}")

# Plot 1: Bar chart of caption lengths

In [ ]:
caption_lengths = [len(cap.split()) for cap in captions]
plt.figure(figsize=(8, 4))
plt.bar([f"Caption {i+1}" for i in range(num_captions)], caption_lengths)
plt.ylabel("Word Count")
plt.title("📊 Caption Lengths")
plt.show()